# 07 · Indian-language evaluation set  (P6)

No public Indian-language deepfake corpus exists, so we build one. Around 500
clips is plenty — this is evidence for one slide, not a research corpus.

**The asymmetry this validates:**

| Signal | Language-dependent? | Why |
|---|---|---|
| synthesis artifacts | **no** | physics — how the waveform was manufactured |
| prosody | **yes** | rhythm and intonation differ completely |
| turn-taking | yes, but cancelled | by the within-call comparison in Branch D |

If artifact detection really is language-agnostic, per-language EER should be
comparable. That is the claim this notebook tests.

In [ ]:
# --- Colab setup -----------------------------------------------------------
# Run this first in every notebook.  Idempotent.
import os, sys, subprocess
from pathlib import Path

IN_COLAB = "google.colab" in sys.modules

if IN_COLAB:
    from google.colab import drive
    drive.mount("/content/drive", force_remount=False)
    # Keep the repo and all caches on Drive so a disconnect does not cost you
    # the feature extraction pass.
    PROJECT = Path("/content/drive/MyDrive/voice-integrity")
    if not PROJECT.exists():
        raise SystemExit(
            f"Upload or clone the repo to {PROJECT} first.\n"
            "  !git clone <your-repo-url> /content/drive/MyDrive/voice-integrity"
        )
else:
    PROJECT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()

os.chdir(PROJECT)
sys.path.insert(0, str(PROJECT / "src"))

# Repo-local model cache.  Set BEFORE importing transformers, or it will use
# the default location and the cache will not be portable to the demo machine.
os.environ["HF_HOME"] = str(PROJECT / "cache" / "huggingface")
os.environ["TORCH_HOME"] = str(PROJECT / "cache" / "torch")
os.environ["HF_HUB_DISABLE_TELEMETRY"] = "1"

print("project:", PROJECT)
print("python :", sys.version.split()[0])

In [ ]:
if IN_COLAB:
    !pip install -q transformers speechbrain soundfile librosa pydantic pyyaml cryptography wandb
    !apt-get -qq install -y ffmpeg libopencore-amrnb-dev > /dev/null

# AMR-NB encoding is the one that silently goes missing.  If this prints
# nothing, your mobile-codec augmentation does nothing and the whole
# codec-robustness result quietly evaporates.
!ffmpeg -hide_banner -encoders 2>/dev/null | grep -i amr || echo "AMR-NB ENCODER MISSING"

## Recipe

```
Real   AI4Bharat Kathbath / IndicSUPERB   real speakers, ~12 languages
Fake   clone THOSE SAME speakers          XTTS-v2, OpenVoice v2, F5-TTS,
                                          AI4Bharat Indic Parler-TTS
Both   through the codec chain            matched pairs, phone conditions
```

Cloning the **same** speakers is what makes this an evaluation set rather than
two unrelated piles of audio: the model cannot succeed by learning speaker
identity or recording conditions instead of synthesis.

Using several generators matters too — if every fake comes from one tool, the
model learns that tool.

In [ ]:
# Cloning tools pull conflicting dependencies.  Install them in a SEPARATE
# environment, or in Colab run this in a dedicated runtime.
if IN_COLAB:
    !pip install -q TTS  # Coqui XTTS-v2

In [ ]:
from vif.common.config import load_config
from vif.data.manifests import build_from_directory, summarise, write_manifest
import json

config = load_config("configs")

real = build_from_directory(
    "data/raw/indian/real", label="bonafide", split="eval",
    corpus="indian", lang="multi",
)
print(f"{len(real)} genuine clips")

In [ ]:
# Clone each reference speaker.  Run in the cloning environment.
from pathlib import Path

CLONE_TEXT = "Please approve the transfer immediately, it is urgent."

def clone_xtts(reference: Path, out_path: Path, language: str = "hi"):
    from TTS.api import TTS
    tts = TTS("tts_models/multilingual/multi-dataset/xtts_v2")
    out_path.parent.mkdir(parents=True, exist_ok=True)
    tts.tts_to_file(
        text=CLONE_TEXT,
        speaker_wav=str(reference),
        language=language,
        file_path=str(out_path),
    )

print("Run clone_xtts over the reference clips, writing to data/raw/indian/fake/")

In [ ]:
# Push both halves through the telephony chain.
from vif.data.augment import CodecAugmenter, check_ffmpeg_codecs
from vif.data.datasets import load_audio
import numpy as np, soundfile as sf

print(check_ffmpeg_codecs(config.augment.codecs))

augmenter = CodecAugmenter(config.augment, config.model.audio.sample_rate, seed=99)

for source, target in [("data/raw/indian/real", "data/raw/indian/real_codec"),
                       ("data/raw/indian/fake", "data/raw/indian/fake_codec")]:
    src, dst = Path(source), Path(target)
    if not src.exists():
        print("skip", source); continue
    dst.mkdir(parents=True, exist_ok=True)
    for path in sorted(src.rglob("*.wav")):
        wav = load_audio(path, config.model.audio.sample_rate)
        degraded, condition = augmenter(wav)
        sf.write(str(dst / f"{path.stem}__{condition}.wav"),
                 np.clip(degraded, -1, 1), config.model.audio.sample_rate)
    print(f"{source} -> {target}")

In [ ]:
# Build the manifest and check the balance.
real_codec = build_from_directory("data/raw/indian/real_codec", label="bonafide",
                                  split="eval", corpus="indian", lang="multi",
                                  condition="codec")
fake_codec = build_from_directory("data/raw/indian/fake_codec", label="spoof",
                                  split="eval", corpus="indian", lang="multi",
                                  attack="xtts_v2", condition="codec")
items = real_codec + fake_codec
write_manifest(items, "data/manifests/indian_eval.jsonl")
print(json.dumps(summarise(items), indent=2))

## Evaluate, per language

The claim under test: artifact detection is language-agnostic, so these
numbers should be comparable to the in-domain result. If one language is
markedly worse, say so and investigate rather than averaging it away.

In [ ]:
from vif.eval.runner import per_language
from vif.models.heads import load_checkpoint
from vif.train.loop import score_manifest
from vif.data.manifests import read_manifest

head, _ = load_checkpoint("models/checkpoints/codec_robust.pt", config.model.head,
                          feat_dim=config.model.frontend.hidden_dim)
items = read_manifest("data/manifests/indian_eval.jsonl")

# Extract features for this set first (notebook 01 pattern), then:
# scores = score_manifest(head, items, "data/features/indian", device=DEVICE)
# _ = per_language(items, scores)
print("Extract features for the Indian set, then run per_language().")

## What to put on the slide

One bar per language, EER on the y-axis, with the in-domain number as a
reference line. Comparable bars support the language-agnostic claim; a
divergent one is a finding worth reporting honestly.